In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

class TrumpShootingEventAnalyzer:
    def __init__(self, data_path='trump_2024_daily_history.csv'):
        self.data_path = data_path
        self.df = None
        self.event_date = pd.to_datetime('2024-07-13')
        self.metrics = {}
        self.shock_index = {}
        self.extreme_volatility_dates = {}
        
    def load_and_preprocess_data(self):
        print("=" * 80)
        print("Data Loading and Preprocessing")
        print("=" * 80)
        
        self.df = pd.read_csv(self.data_path)
        self.df['date'] = pd.to_datetime(self.df['date']).dt.date
        self.df['date'] = pd.to_datetime(self.df['date'])
        self.df.set_index('date', inplace=True)
        self.df.sort_index(inplace=True)
        
        # Check if event date exists in data
        if self.event_date not in self.df.index:
            print(f"Warning: Event date {self.event_date.date()} not in data")
            available_dates = self.df.index.tolist()
            closest_date = min(available_dates, key=lambda x: abs(x - self.event_date))
            print(f"Using closest date: {closest_date.date()}")
            self.event_date = closest_date
        
        print(f"Data range: {self.df.index.min().date()} to {self.df.index.max().date()}")
        print(f"Total data points: {len(self.df)}")
        print(f"Event date: {self.event_date.date()}")
        print(f"Event day price: ${self.df.loc[self.event_date, 'price']:.4f}")
        
        return self.df
    
    def calculate_technical_indicators(self):
        """Calculate technical indicators"""
        print("\n" + "=" * 80)
        print("Technical Indicators Calculation")
        print("=" * 80)
        
        # 1. Daily returns
        self.df['daily_return_pct'] = self.df['price'].pct_change() * 100
        
        # 2. 20-day historical volatility (rolling std)
        self.df['volatility_20d'] = self.df['daily_return_pct'].rolling(window=20, min_periods=10).std()
        
        # 3. Moving averages
        self.df['sma_10'] = self.df['price'].rolling(window=10, min_periods=5).mean()
        self.df['sma_30'] = self.df['price'].rolling(window=30, min_periods=15).mean()
        
        # 4. Relative Strength Index (RSI, 14-day)
        delta = self.df['price'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        self.df['rsi'] = 100 - (100 / (1 + rs))
        
        # 5. Bollinger Bands
        self.df['bb_middle'] = self.df['price'].rolling(window=20).mean()
        bb_std = self.df['price'].rolling(window=20).std()
        self.df['bb_upper'] = self.df['bb_middle'] + (bb_std * 2)
        self.df['bb_lower'] = self.df['bb_middle'] - (bb_std * 2)
        
        print("✓ Daily returns calculated")
        print("✓ 20-day volatility calculated")
        print("✓ Moving averages calculated")
        print("✓ RSI indicator calculated")
        print("✓ Bollinger Bands calculated")
        
        return self.df
    
    def calculate_event_metrics(self, window_days=10):
        """Calculate key metrics on the event day"""
        print("\n" + "=" * 80)
        print("Event Day Key Metrics Calculation")
        print("=" * 80)
        
        target_date = self.event_date
        
        # Get dates before and after target
        all_dates = self.df.index.tolist()
        target_idx = all_dates.index(target_date)
        
        # Get previous and next day
        prev_day = all_dates[target_idx-1] if target_idx > 0 else None
        next_day = all_dates[target_idx+1] if target_idx < len(all_dates)-1 else None
        
        # Get event day return
        event_day_return = self.df.loc[target_date, 'daily_return_pct']
        
        # Calculate max reaction within 1-3 trading days after event
        max_reaction_pct = 0
        max_reaction_day = None
        for i in range(1, min(4, len(all_dates)-target_idx)):
            reaction_date = all_dates[target_idx + i]
            reaction_pct = self.df.loc[reaction_date, 'daily_return_pct']
            if abs(reaction_pct) > abs(max_reaction_pct):
                max_reaction_pct = reaction_pct
                max_reaction_day = reaction_date
        
        # Define window boundaries
        window_start = max(self.df.index[0], target_date - timedelta(days=window_days))
        window_end = min(self.df.index[-1], target_date + timedelta(days=window_days))
        
        # Get window data
        window_data = self.df.loc[window_start:window_end]
        
        # Calculate trends
        trend_before = 0
        if len(window_data.loc[:target_date]) > 1:
            first_price = window_data.loc[:target_date]['price'].iloc[0]
            event_price = window_data.loc[target_date, 'price']
            trend_before = ((event_price - first_price) / first_price) * 100
        
        trend_after = 0
        if len(window_data.loc[target_date:]) > 1:
            last_price = window_data.loc[target_date:]['price'].iloc[-1]
            event_price = window_data.loc[target_date, 'price']
            trend_after = ((last_price - event_price) / event_price) * 100
        
        # Calculate historical average volatility
        historical_vol = self.df['volatility_20d'].dropna().mean()
        volatility_multiple = 0
        if historical_vol > 0:
            event_vol = self.df.loc[target_date, 'volatility_20d']
            if not pd.isna(event_vol):
                volatility_multiple = event_vol / historical_vol
        
        # Calculate deviation from moving averages
        price_vs_sma10 = None
        if not pd.isna(self.df.loc[target_date, 'sma_10']):
            price_vs_sma10 = ((self.df.loc[target_date, 'price'] - self.df.loc[target_date, 'sma_10']) / 
                            self.df.loc[target_date, 'sma_10']) * 100
        
        price_vs_sma30 = None
        if not pd.isna(self.df.loc[target_date, 'sma_30']):
            price_vs_sma30 = ((self.df.loc[target_date, 'price'] - self.df.loc[target_date, 'sma_30']) / 
                            self.df.loc[target_date, 'sma_30']) * 100
        
        # Calculate RSI status
        rsi_value = self.df.loc[target_date, 'rsi'] if 'rsi' in self.df.columns and not pd.isna(self.df.loc[target_date, 'rsi']) else None
        rsi_status = "Neutral"
        if rsi_value:
            if rsi_value > 70:
                rsi_status = "Overbought"
            elif rsi_value < 30:
                rsi_status = "Oversold"
        
        # Calculate Bollinger Band position
        bb_position = None
        if 'bb_upper' in self.df.columns and not pd.isna(self.df.loc[target_date, 'bb_upper']):
            price = self.df.loc[target_date, 'price']
            bb_upper = self.df.loc[target_date, 'bb_upper']
            bb_lower = self.df.loc[target_date, 'bb_lower']
            if price > bb_upper:
                bb_position = "Above upper band (overbought)"
            elif price < bb_lower:
                bb_position = "Below lower band (oversold)"
            else:
                bb_position = "Within bands"
        
        # Save metrics
        self.metrics = {
            'event_date': target_date.strftime('%Y-%m-%d'),
            'price_on_event': self.df.loc[target_date, 'price'],
            'price_previous_day': self.df.loc[prev_day, 'price'] if prev_day else None,
            'price_next_day': self.df.loc[next_day, 'price'] if next_day else None,
            
            # Return metrics
            'event_day_return_pct': event_day_return,
            'max_reaction_after_event_pct': max_reaction_pct,
            'max_reaction_date': max_reaction_day.strftime('%Y-%m-%d') if max_reaction_day else None,
            
            # Volatility metrics
            'volatility_20d_on_event': self.df.loc[target_date, 'volatility_20d'],
            'volatility_multiple': volatility_multiple,
            'avg_volatility_in_window': window_data['volatility_20d'].mean(),
            
            # Trend metrics
            f'trend_before_{window_days}d_pct': trend_before,
            f'trend_after_{window_days}d_pct': trend_after,
            'trend_change_abs_pct': abs(trend_after - trend_before),
            
            # Technical indicators
            'deviation_from_sma10_pct': price_vs_sma10,
            'deviation_from_sma30_pct': price_vs_sma30,
            'rsi_value': rsi_value,
            'rsi_status': rsi_status,
            'bollinger_band_position': bb_position,
            
            # Window info
            'event_window_start': window_start.strftime('%Y-%m-%d'),
            'event_window_end': window_end.strftime('%Y-%m-%d'),
            'window_data_points': len(window_data),
        }
        
        # Print key metrics
        print(f"Event date: {self.metrics['event_date']}")
        print(f"Price on event: ${self.metrics['price_on_event']:.4f}")
        if self.metrics['price_previous_day']:
            print(f"Price previous day: ${self.metrics['price_previous_day']:.4f}")
        print(f"Event day return: {self.metrics['event_day_return_pct']:.2f}%")
        print(f"Max reaction after event: {self.metrics['max_reaction_after_event_pct']:.2f}% (date: {self.metrics['max_reaction_date']})")
        print(f"20-day volatility: {self.metrics['volatility_20d_on_event']:.4f}")
        print(f"Volatility multiple: {self.metrics['volatility_multiple']:.2f}x")
        
        return self.metrics
    
    def calculate_shock_index(self, window=5):
        """Calculate event shock index"""
        print("\n" + "=" * 80)
        print("Event Shock Index Calculation")
        print("=" * 80)
        
        target_date = self.event_date
        
        if target_date not in self.df.index:
            print("Event date not in data, cannot calculate shock index")
            return None
        
        target_idx = self.df.index.get_loc(target_date)
        
        # Calculate pre- and post-event windows
        pre_start = max(0, target_idx - window)
        pre_window = self.df.iloc[pre_start:target_idx]
        
        post_end = min(len(self.df), target_idx + window + 1)
        post_window = self.df.iloc[target_idx+1:post_end]
        
        if len(pre_window) == 0 or len(post_window) == 0:
            print("Insufficient window data, cannot calculate shock index")
            return None
        
        # 1. Price shock (average price change)
        pre_avg_price = pre_window['price'].mean()
        post_avg_price = post_window['price'].mean()
        price_shock = ((post_avg_price - pre_avg_price) / pre_avg_price) * 100
        
        # 2. Volatility shock
        pre_avg_vol = pre_window['volatility_20d'].mean()
        post_avg_vol = post_window['volatility_20d'].mean()
        vol_shock = 0
        if pre_avg_vol > 0:
            vol_shock = ((post_avg_vol - pre_avg_vol) / pre_avg_vol) * 100
        
        # 3. Maximum return shock (absolute)
        max_return_shock = post_window['daily_return_pct'].abs().max()
        
        # 4. Trend shock (direction change)
        pre_trend = ((pre_window['price'].iloc[-1] - pre_window['price'].iloc[0]) / 
                    pre_window['price'].iloc[0]) * 100 if len(pre_window) > 1 else 0
        post_trend = ((post_window['price'].iloc[-1] - post_window['price'].iloc[0]) / 
                     post_window['price'].iloc[0]) * 100 if len(post_window) > 1 else 0
        trend_shock = abs(post_trend - pre_trend)
        
        # Composite shock index (weighted)
        # Weights: price shock 40%, volatility shock 25%, max return 20%, trend shock 15%
        composite_shock = (
            abs(price_shock) * 0.40 +
            abs(vol_shock) * 0.25 +
            abs(max_return_shock) * 0.20 +
            trend_shock * 0.15
        )
        
        self.shock_index = {
            'price_shock_pct': price_shock,
            'volatility_shock_pct': vol_shock,
            'max_return_shock_pct': max_return_shock,
            'trend_shock_pct': trend_shock,
            'composite_shock_index': composite_shock,
            'window_days': window,
            'pre_window_start': pre_window.index[0].strftime('%Y-%m-%d'),
            'post_window_end': post_window.index[-1].strftime('%Y-%m-%d'),
        }
        
        # Print results
        print(f"Price shock: {price_shock:.2f}%")
        print(f"Volatility shock: {vol_shock:.2f}%")
        print(f"Max return shock: {max_return_shock:.2f}%")
        print(f"Trend shock: {trend_shock:.2f}%")
        print(f"Composite shock index: {composite_shock:.2f}")
        
        return self.shock_index
    
    def classify_event_severity(self):
        """Classify event severity (revised)"""
        print("\n" + "=" * 80)
        print("Event Severity Classification (Revised)")
        print("=" * 80)
        
        if not self.metrics:
            print("Please calculate event metrics first")
            return None
        
        # Get key metrics
        immediate_return = abs(self.metrics['event_day_return_pct'])
        max_reaction = abs(self.metrics['max_reaction_after_event_pct'])
        vol_multiple = self.metrics['volatility_multiple']
        
        # Stage 1: Immediate impact assessment
        if immediate_return < 1:
            immediate_severity = "Minimal"
            immediate_reason = f"Event day fluctuation minimal: {immediate_return:.2f}%"
        elif immediate_return < 2:
            immediate_severity = "Slight"
            immediate_reason = f"Event day slight fluctuation: {immediate_return:.2f}%"
        elif immediate_return < 5:
            immediate_severity = "Moderate"
            immediate_reason = f"Event day moderate fluctuation: {immediate_return:.2f}%"
        elif immediate_return < 10:
            immediate_severity = "Significant"
            immediate_reason = f"Event day significant fluctuation: {immediate_return:.2f}%"
        else:
            immediate_severity = "Major"
            immediate_reason = f"Event day major fluctuation: {immediate_return:.2f}%"
        
        # Stage 2: Overall impact assessment (considering delayed reaction)
        if max_reaction < 5:
            overall_severity = "Minor Impact"
            overall_reason = f"Limited maximum reaction: {max_reaction:.2f}%"
        elif max_reaction < 10:
            overall_severity = "Moderate Impact"
            overall_reason = f"Notable reaction: {max_reaction:.2f}%"
        elif max_reaction < 15:
            overall_severity = "Significant Impact"
            overall_reason = f"Strong reaction: {max_reaction:.2f}%"
        elif max_reaction < 20:
            overall_severity = "Major Impact"
            overall_reason = f"Major reaction: {max_reaction:.2f}%"
        else:
            overall_severity = "Extreme Impact"
            overall_reason = f"Extreme reaction: {max_reaction:.2f}%"
        
        # Volatility multiple assessment
        if vol_multiple < 0.5:
            vol_assessment = "Very low volatility"
        elif vol_multiple < 1:
            vol_assessment = "Below average volatility"
        elif vol_multiple < 2:
            vol_assessment = "Normal volatility"
        elif vol_multiple < 3:
            vol_assessment = "Elevated volatility"
        else:
            vol_assessment = "Extreme volatility"
        
        # Trend change assessment
        trend_change = self.metrics['trend_change_abs_pct']
        if trend_change < 5:
            trend_assessment = "Slight trend change"
        elif trend_change < 10:
            trend_assessment = "Notable trend change"
        elif trend_change < 20:
            trend_assessment = "Significant trend change"
        else:
            trend_assessment = "Trend reversal"
        
        # Combined assessment
        if immediate_severity == "Minimal" and overall_severity == "Significant Impact":
            final_assessment = "Delayed Reaction Significant Event"
            final_reason = "Event day reaction minimal, but subsequent reaction significant"
        elif immediate_severity == "Slight" and overall_severity == "Major Impact":
            final_assessment = "Delayed Reaction Major Event"
            final_reason = "Event day reaction limited, but subsequent reaction major"
        elif immediate_severity in ["Significant", "Major"]:
            final_assessment = "Immediate Reaction " + overall_severity
            final_reason = "Significant reaction on event day"
        else:
            final_assessment = overall_severity
            final_reason = f"Overall impact level: {overall_severity}"
        
        # Save classification results
        self.severity_assessment = {
            'event_day_impact': immediate_severity,
            'event_day_reason': immediate_reason,
            'overall_impact': overall_severity,
            'overall_reason': overall_reason,
            'volatility_assessment': vol_assessment,
            'trend_change_assessment': trend_assessment,
            'combined_assessment': final_assessment,
            'combined_reason': final_reason,
        }
        
        # Print results
        print(f"Event day impact: {immediate_severity}")
        print(f"Reason: {immediate_reason}")
        print(f"\nOverall impact: {overall_severity}")
        print(f"Reason: {overall_reason}")
        print(f"\nVolatility assessment: {vol_assessment} (multiple: {vol_multiple:.2f}x)")
        print(f"Trend change assessment: {trend_assessment} (magnitude: {trend_change:.2f}%)")
        print(f"\nCombined assessment: {final_assessment}")
        print(f"Reason: {final_reason}")
        
        return self.severity_assessment
    
    def detect_abnormal_volatility(self):
        """Detect abnormal volatility"""
        print("\n" + "=" * 80)
        print("Abnormal Volatility Detection")
        print("=" * 80)
        
        target_date = self.event_date
        
        # Get historical data (before event)
        historical_data = self.df.loc[:target_date - timedelta(days=1)]
        
        # Calculate historical statistics
        hist_returns = historical_data['daily_return_pct'].dropna()
        hist_std = hist_returns.std()
        hist_avg_vol = historical_data['volatility_20d'].dropna().mean()
        
        # Event day metrics
        event_return = self.metrics['event_day_return_pct']
        event_return_abs = abs(event_return)
        
        # Test 1: 2 standard deviation rule
        threshold_2sigma = 2 * hist_std
        sigma_detection = "Pass" if event_return_abs > threshold_2sigma else "Fail"
        
        # Test 2: 3x volatility rule
        threshold_3vol = 3 * hist_avg_vol
        vol_detection = "Pass" if event_return_abs > threshold_3vol else "Fail"
        
        # Test 3: Percentile rank
        all_returns_abs = self.df['daily_return_pct'].abs().dropna()
        rank = (all_returns_abs >= event_return_abs).sum()
        total = len(all_returns_abs)
        percentile = (rank / total) * 100
        
        # Test 4: Compare with historical max volatility
        max_historical_return = hist_returns.abs().max()
        comparison_to_max = event_return_abs / max_historical_return if max_historical_return > 0 else 0
        
        # Save detection results
        self.abnormal_detection = {
            'historical_return_std': hist_std,
            'historical_avg_volatility': hist_avg_vol,
            '2sigma_threshold_pct': threshold_2sigma,
            '2sigma_test': sigma_detection,
            '3x_vol_threshold_pct': threshold_3vol,
            '3x_vol_test': vol_detection,
            'event_return_abs_rank': f"{rank}/{total}",
            'percentile_rank_pct': percentile,
            'ratio_to_historical_max': f"{comparison_to_max:.2f}x",
            'is_abnormal': "Yes" if (sigma_detection == "Pass" or vol_detection == "Pass" or percentile <= 10) else "No",
        }
        
        # Print results
        print(f"Historical return standard deviation: {hist_std:.2f}%")
        print(f"Historical average volatility: {hist_avg_vol:.4f}")
        print(f"\nTest 1 - 2 standard deviation rule:")
        print(f"  Threshold: {threshold_2sigma:.2f}%")
        print(f"  Event day return: {event_return:.2f}%")
        print(f"  Result: {sigma_detection}")
        
        print(f"\nTest 2 - 3x volatility rule:")
        print(f"  Threshold: {threshold_3vol:.2f}%")
        print(f"  Event day return: {event_return:.2f}%")
        print(f"  Result: {vol_detection}")
        
        print(f"\nTest 3 - Percentile rank:")
        print(f"  Rank: {rank}/{total}")
        print(f"  Percentile: {percentile:.1f}%")
        
        print(f"\nTest 4 - Comparison with historical max:")
        print(f"  Historical max: {max_historical_return:.2f}%")
        print(f"  Event/historical max ratio: {comparison_to_max:.2f}x")
        
        print(f"\nOverall judgment: Event day volatility is {'abnormal' if self.abnormal_detection['is_abnormal'] == 'Yes' else 'normal'}")
        
        return self.abnormal_detection
    
    def find_extreme_volatility_dates(self):
        """Find the most volatile dates in the dataset and compare with event date"""
        print("\n" + "=" * 80)
        print("Extreme Volatility Date Analysis")
        print("=" * 80)
        
        # Ensure daily returns are calculated
        if 'daily_return_pct' not in self.df.columns:
            self.df['daily_return_pct'] = self.df['price'].pct_change() * 100
        
        self.df['abs_daily_return'] = self.df['daily_return_pct'].abs()
        
        # 1. Find top 10 most volatile days by absolute return
        top_10_volatile = self.df.nlargest(10, 'abs_daily_return')
        
        # 2. Find largest single-day gain and loss
        max_gain = self.df['daily_return_pct'].max()
        max_gain_date = self.df['daily_return_pct'].idxmax()
        
        max_loss = self.df['daily_return_pct'].min()
        max_loss_date = self.df['daily_return_pct'].idxmin()
        
        # 3. Find day with largest absolute return
        max_abs_return = self.df['abs_daily_return'].max()
        max_abs_date = self.df['abs_daily_return'].idxmax()
        max_abs_value = self.df.loc[max_abs_date, 'daily_return_pct']
        
        # 4. Find event day ranking
        event_return_abs = abs(self.df.loc[self.event_date, 'daily_return_pct'])
        event_rank = (self.df['abs_daily_return'] >= event_return_abs).sum()
        total_days = len(self.df['abs_daily_return'].dropna())
        event_percentile = (event_rank / total_days) * 100
        
        print("Top 10 most volatile days in dataset:")
        print("-" * 80)
        for i, (date, row) in enumerate(top_10_volatile.iterrows(), 1):
            is_event_day = "★" if date == self.event_date else " "
            # Handle volatility string
            if 'volatility_20d' in row and not pd.isna(row['volatility_20d']):
                vol_str = f"{row['volatility_20d']:6.4f}"
            else:
                vol_str = "N/A"
            print(f"{i:2}{is_event_day} {date.date()} - "
                  f"Return: {row['daily_return_pct']:7.2f}%, "
                  f"Price: ${row['price']:7.4f}, "
                  f"Volatility: {vol_str:>8}")
        
        print("\n" + "=" * 60)
        print("Extreme Volatility Day Details:")
        print("=" * 60)
        
        print(f"Largest single-day gain:")
        print(f"  Date: {max_gain_date.date()}")
        print(f"  Gain: {max_gain:.2f}%")
        print(f"  Price: ${self.df.loc[max_gain_date, 'price']:.4f}")
        
        # Get previous and next prices
        date_idx = self.df.index.get_loc(max_gain_date)
        if date_idx > 0:
            prev_price = self.df.iloc[date_idx-1]['price']
            print(f"  Previous day price: ${prev_price:.4f}")
        if date_idx < len(self.df)-1:
            next_price = self.df.iloc[date_idx+1]['price']
            print(f"  Next day price: ${next_price:.4f}")
        
        print(f"\nLargest single-day loss:")
        print(f"  Date: {max_loss_date.date()}")
        print(f"  Loss: {max_loss:.2f}%")
        print(f"  Price: ${self.df.loc[max_loss_date, 'price']:.4f}")
        
        # Get previous and next prices
        date_idx = self.df.index.get_loc(max_loss_date)
        if date_idx > 0:
            prev_price = self.df.iloc[date_idx-1]['price']
            print(f"  Previous day price: ${prev_price:.4f}")
        if date_idx < len(self.df)-1:
            next_price = self.df.iloc[date_idx+1]['price']
            print(f"  Next day price: ${next_price:.4f}")
        
        print(f"\nLargest absolute return:")
        print(f"  Date: {max_abs_date.date()}")
        print(f"  Return: {max_abs_value:.2f}%")
        print(f"  Price: ${self.df.loc[max_abs_date, 'price']:.4f}")
        
        print("\n" + "=" * 60)
        print(f"Event day ({self.event_date.date()}) volatility ranking:")
        print("=" * 60)
        print(f"  Return: {self.df.loc[self.event_date, 'daily_return_pct']:.2f}%")
        print(f"  Absolute rank: {event_rank}/{total_days}")
        print(f"  Percentile: {event_percentile:.1f}%")
        
        # Analyze if event day is in top 10
        if self.event_date in top_10_volatile.index:
            rank_in_top10 = list(top_10_volatile.index).index(self.event_date) + 1
            print(f"\n★ Event day ranks #{rank_in_top10} among extreme volatility days")
        else:
            print(f"\n★ Event day is not among the top 10 extreme volatility days")
        
        # Quarterly volatility analysis
        self.df['quarter'] = self.df.index.to_period('Q')
        quarterly_stats = self.df.groupby('quarter').agg({
            'abs_daily_return': ['mean', 'max', 'std'],
            'daily_return_pct': ['mean', 'min', 'max']
        }).round(2)
        
        print("\n" + "=" * 60)
        print("Quarterly Volatility Statistics:")
        print("=" * 60)
        print(quarterly_stats)
        
        # Save results
        self.extreme_volatility_dates = {
            'max_gain_date': max_gain_date.strftime('%Y-%m-%d'),
            'max_gain_pct': max_gain,
            'max_loss_date': max_loss_date.strftime('%Y-%m-%d'),
            'max_loss_pct': max_loss,
            'max_abs_return_date': max_abs_date.strftime('%Y-%m-%d'),
            'max_abs_return_pct': max_abs_value,
            'event_day_rank': event_rank,
            'event_day_percentile_pct': event_percentile,
            'event_day_in_top10': self.event_date in top_10_volatile.index,
            'event_day_rank_in_top10': list(top_10_volatile.index).index(self.event_date) + 1 if self.event_date in top_10_volatile.index else None,
            'top10_dates': [date.strftime('%Y-%m-%d') for date in top_10_volatile.index],
        }
        
        # Save to CSV
        top_10_volatile.to_csv('top_10_volatile_days.csv')
        print("\n✓ Top 10 volatile days saved to: top_10_volatile_days.csv")
        
        return self.extreme_volatility_dates
    
    def analyze_event_context(self):
        """Analyze market context before and after the event"""
        print("\n" + "=" * 80)
        print("Event Context Analysis")
        print("=" * 80)
        
        target_date = self.event_date
        
        # 30 days before event
        pre_window_start = target_date - timedelta(days=30)
        pre_window = self.df.loc[pre_window_start:target_date - timedelta(days=1)]
        
        # 30 days after event
        post_window_end = target_date + timedelta(days=30)
        post_window = self.df.loc[target_date + timedelta(days=1):post_window_end]
        
        # Calculate key statistics
        pre_stats = {
            'avg_price': pre_window['price'].mean(),
            'price_std': pre_window['price'].std(),
            'avg_return_pct': pre_window['daily_return_pct'].mean(),
            'return_volatility': pre_window['daily_return_pct'].std(),
            'avg_volatility_20d': pre_window['volatility_20d'].mean(),
            'trend_direction': "Up" if pre_window['price'].iloc[-1] > pre_window['price'].iloc[0] else "Down",
            'trend_magnitude_pct': ((pre_window['price'].iloc[-1] - pre_window['price'].iloc[0]) / pre_window['price'].iloc[0]) * 100,
        }
        
        post_stats = {
            'avg_price': post_window['price'].mean(),
            'price_std': post_window['price'].std(),
            'avg_return_pct': post_window['daily_return_pct'].mean(),
            'return_volatility': post_window['daily_return_pct'].std(),
            'avg_volatility_20d': post_window['volatility_20d'].mean(),
            'trend_direction': "Up" if post_window['price'].iloc[-1] > post_window['price'].iloc[0] else "Down",
            'trend_magnitude_pct': ((post_window['price'].iloc[-1] - post_window['price'].iloc[0]) / post_window['price'].iloc[0]) * 100,
        }
        
        # Calculate changes
        changes = {
            'price_change_pct': ((post_stats['avg_price'] - pre_stats['avg_price']) / pre_stats['avg_price']) * 100,
            'volatility_change_pct': ((post_stats['avg_volatility_20d'] - pre_stats['avg_volatility_20d']) / pre_stats['avg_volatility_20d']) * 100,
            'trend_direction_change': "Reversal" if pre_stats['trend_direction'] != post_stats['trend_direction'] else "Continuation",
        }
        
        # Save results
        self.context_analysis = {
            'pre_event_market': pre_stats,
            'post_event_market': post_stats,
            'event_impact_changes': changes,
            'analysis_window': "30 days before and after event",
        }
        
        # Print results
        print("Market conditions 30 days before event:")
        print(f"  Avg price: ${pre_stats['avg_price']:.4f}")
        print(f"  Avg return: {pre_stats['avg_return_pct']:.2f}%")
        print(f"  Avg volatility: {pre_stats['avg_volatility_20d']:.4f}")
        print(f"  Trend: {pre_stats['trend_direction']} {abs(pre_stats['trend_magnitude_pct']):.2f}%")
        
        print("\nMarket conditions 30 days after event:")
        print(f"  Avg price: ${post_stats['avg_price']:.4f}")
        print(f"  Avg return: {post_stats['avg_return_pct']:.2f}%")
        print(f"  Avg volatility: {post_stats['avg_volatility_20d']:.4f}")
        print(f"  Trend: {post_stats['trend_direction']} {abs(post_stats['trend_magnitude_pct']):.2f}%")
        
        print("\nEvent impact changes:")
        print(f"  Price change: {changes['price_change_pct']:.2f}%")
        print(f"  Volatility change: {changes['volatility_change_pct']:.2f}%")
        print(f"  Trend direction: {changes['trend_direction_change']}")
        
        return self.context_analysis
    
    def generate_summary_statistics(self):
        """Generate summary statistics"""
        print("\n" + "=" * 80)
        print("Summary Statistics")
        print("=" * 80)
        
        # Full dataset statistics
        full_stats = {
            'date_range': f"{self.df.index.min().date()} to {self.df.index.max().date()}",
            'data_points': len(self.df),
            'avg_price': self.df['price'].mean(),
            'median_price': self.df['price'].median(),
            'price_std': self.df['price'].std(),
            'min_price': self.df['price'].min(),
            'max_price': self.df['price'].max(),
            'avg_daily_return_pct': self.df['daily_return_pct'].mean(),
            'daily_return_std': self.df['daily_return_pct'].std(),
            'positive_return_days': (self.df['daily_return_pct'] > 0).sum(),
            'negative_return_days': (self.df['daily_return_pct'] < 0).sum(),
            'zero_return_days': (self.df['daily_return_pct'] == 0).sum(),
            'avg_volatility_20d': self.df['volatility_20d'].mean(),
        }
        
        # Comparison before and after event (60 days each)
        event_idx = self.df.index.get_loc(self.event_date)
        pre_start = max(0, event_idx - 60)
        post_end = min(len(self.df), event_idx + 60)
        
        pre_period = self.df.iloc[pre_start:event_idx]
        post_period = self.df.iloc[event_idx+1:post_end]
        
        comparison = {
            'pre_avg_price': pre_period['price'].mean(),
            'post_avg_price': post_period['price'].mean(),
            'price_change_pct': ((post_period['price'].mean() - pre_period['price'].mean()) / pre_period['price'].mean()) * 100,
            'pre_avg_volatility': pre_period['volatility_20d'].mean(),
            'post_avg_volatility': post_period['volatility_20d'].mean(),
            'volatility_change_pct': ((post_period['volatility_20d'].mean() - pre_period['volatility_20d'].mean()) / pre_period['volatility_20d'].mean()) * 100,
            'pre_positive_return_pct': (pre_period['daily_return_pct'] > 0).sum() / len(pre_period) * 100,
            'post_positive_return_pct': (post_period['daily_return_pct'] > 0).sum() / len(post_period) * 100,
        }
        
        # Save results
        self.summary_stats = {
            'dataset_stats': full_stats,
            'pre_post_comparison': comparison,
        }
        
        # Print results
        print("Full dataset statistics:")
        print(f"  Date range: {full_stats['date_range']}")
        print(f"  Data points: {full_stats['data_points']}")
        print(f"  Avg price: ${full_stats['avg_price']:.4f}")
        print(f"  Price range: ${full_stats['min_price']:.4f} - ${full_stats['max_price']:.4f}")
        print(f"  Avg daily return: {full_stats['avg_daily_return_pct']:.2f}%")
        print(f"  Daily return std: {full_stats['daily_return_std']:.2f}%")
        print(f"  Positive return days: {full_stats['positive_return_days']} ({full_stats['positive_return_days']/full_stats['data_points']*100:.1f}%)")
        print(f"  Negative return days: {full_stats['negative_return_days']} ({full_stats['negative_return_days']/full_stats['data_points']*100:.1f}%)")
        
        print("\nPre- vs post-event comparison (60 days each):")
        print(f"  Pre-event avg price: ${comparison['pre_avg_price']:.4f}")
        print(f"  Post-event avg price: ${comparison['post_avg_price']:.4f}")
        print(f"  Price change: {comparison['price_change_pct']:.2f}%")
        print(f"  Pre-event avg volatility: {comparison['pre_avg_volatility']:.4f}")
        print(f"  Post-event avg volatility: {comparison['post_avg_volatility']:.4f}")
        print(f"  Volatility change: {comparison['volatility_change_pct']:.2f}%")
        print(f"  Pre-event positive return proportion: {comparison['pre_positive_return_pct']:.1f}%")
        print(f"  Post-event positive return proportion: {comparison['post_positive_return_pct']:.1f}%")
        
        return self.summary_stats
    
    def save_results(self):
        """Save analysis results to files"""
        print("\n" + "=" * 80)
        print("Saving Analysis Results")
        print("=" * 80)
        
        # Save full dataset with indicators
        self.df.to_csv('trump_full_dataset_with_indicators.csv')
        print("✓ Full dataset saved to: trump_full_dataset_with_indicators.csv")
        
        # Save event day metrics
        if self.metrics:
            metrics_df = pd.DataFrame([self.metrics])
            metrics_df.to_csv('trump_event_metrics.csv', index=False)
            print("✓ Event metrics saved to: trump_event_metrics.csv")
        
        # Save shock index
        if self.shock_index:
            shock_df = pd.DataFrame([self.shock_index])
            shock_df.to_csv('trump_shock_index.csv', index=False)
            print("✓ Shock index saved to: trump_shock_index.csv")
        
        # Save severity assessment
        if hasattr(self, 'severity_assessment'):
            severity_df = pd.DataFrame([self.severity_assessment])
            severity_df.to_csv('trump_severity_assessment.csv', index=False)
            print("✓ Severity assessment saved to: trump_severity_assessment.csv")
        
        # Save abnormal volatility detection
        if hasattr(self, 'abnormal_detection'):
            abnormal_df = pd.DataFrame([self.abnormal_detection])
            abnormal_df.to_csv('trump_abnormal_detection.csv', index=False)
            print("✓ Abnormal volatility detection saved to: trump_abnormal_detection.csv")
        
        # Save extreme volatility dates analysis
        if hasattr(self, 'extreme_volatility_dates'):
            extreme_df = pd.DataFrame([self.extreme_volatility_dates])
            extreme_df.to_csv('trump_extreme_volatility_dates.csv', index=False)
            print("✓ Extreme volatility dates saved to: trump_extreme_volatility_dates.csv")
        
        # Save event context analysis
        if hasattr(self, 'context_analysis'):
            # Flatten nested dictionary
            context_data = {}
            for key, value in self.context_analysis.items():
                if isinstance(value, dict):
                    for sub_key, sub_value in value.items():
                        context_data[f"{key}_{sub_key}"] = sub_value
                else:
                    context_data[key] = value
            
            context_df = pd.DataFrame([context_data])
            context_df.to_csv('trump_context_analysis.csv', index=False)
            print("✓ Event context analysis saved to: trump_context_analysis.csv")
        
        # Save summary statistics
        if hasattr(self, 'summary_stats'):
            # Flatten nested dictionary
            summary_data = {}
            for main_key, sub_dict in self.summary_stats.items():
                for sub_key, sub_value in sub_dict.items():
                    summary_data[f"{main_key}_{sub_key}"] = sub_value
            
            summary_df = pd.DataFrame([summary_data])
            summary_df.to_csv('trump_summary_statistics.csv', index=False)
            print("✓ Summary statistics saved to: trump_summary_statistics.csv")
        
        # Generate comprehensive report
        self.generate_comprehensive_report()
        
        return True
    
    def generate_comprehensive_report(self):
        """Generate comprehensive analysis report"""
        print("\n" + "=" * 80)
        print("Generating Comprehensive Report")
        print("=" * 80)
        
        report_lines = []
        report_lines.append("=" * 80)
        report_lines.append("Trump Shooting Event Quantitative Analysis Report")
        report_lines.append("=" * 80)
        report_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report_lines.append(f"Event Date: {self.event_date.strftime('%Y-%m-%d')}")
        report_lines.append("")
        
        # 1. Key findings
        report_lines.append("I. Key Findings")
        report_lines.append("-" * 40)
        
        if self.metrics:
            event_return = self.metrics['event_day_return_pct']
            max_reaction = self.metrics['max_reaction_after_event_pct']
            trend_change = self.metrics['trend_change_abs_pct']
            
            report_lines.append(f"1. Event day return: {event_return:.2f}%")
            report_lines.append(f"2. Max reaction after event: {max_reaction:.2f}%")
            report_lines.append(f"3. Trend change magnitude: {trend_change:.2f}%")
            
            if abs(event_return) < 2 and abs(max_reaction) > 10:
                report_lines.append("4. Market characteristic: Delayed reaction - limited immediate reaction but strong subsequent reaction")
            elif abs(event_return) > 10:
                report_lines.append("4. Market characteristic: Immediate reaction - strong reaction on event day")
        
        # 2. Extreme volatility analysis
        report_lines.append("")
        report_lines.append("II. Extreme Volatility Analysis")
        report_lines.append("-" * 40)
        
        if hasattr(self, 'extreme_volatility_dates'):
            max_gain = self.extreme_volatility_dates.get('max_gain_pct', 0)
            max_gain_date = self.extreme_volatility_dates.get('max_gain_date', 'N/A')
            max_loss = self.extreme_volatility_dates.get('max_loss_pct', 0)
            max_loss_date = self.extreme_volatility_dates.get('max_loss_date', 'N/A')
            event_percentile = self.extreme_volatility_dates.get('event_day_percentile_pct', 0)
            
            report_lines.append(f"Largest single-day gain: {max_gain:.2f}% (Date: {max_gain_date})")
            report_lines.append(f"Largest single-day loss: {max_loss:.2f}% (Date: {max_loss_date})")
            report_lines.append(f"Event day volatility percentile: {event_percentile:.1f}%")
            
            if event_percentile <= 10:
                report_lines.append(f"Event day volatility rank: Extreme (top {event_percentile:.1f}%)")
            elif event_percentile <= 20:
                report_lines.append(f"Event day volatility rank: High (top {event_percentile:.1f}%)")
            elif event_percentile <= 50:
                report_lines.append(f"Event day volatility rank: Moderate (top {event_percentile:.1f}%)")
            else:
                report_lines.append(f"Event day volatility rank: Normal (top {event_percentile:.1f}%)")
        
        # 3. Event impact assessment
        report_lines.append("")
        report_lines.append("III. Event Impact Assessment")
        report_lines.append("-" * 40)
        
        if hasattr(self, 'severity_assessment'):
            report_lines.append(f"Event day impact: {self.severity_assessment['event_day_impact']}")
            report_lines.append(f"Overall impact: {self.severity_assessment['overall_impact']}")
            report_lines.append(f"Combined assessment: {self.severity_assessment['combined_assessment']}")
        
        # 4. Shock index analysis
        report_lines.append("")
        report_lines.append("IV. Shock Index Analysis")
        report_lines.append("-" * 40)
        
        if self.shock_index:
            report_lines.append(f"Composite shock index: {self.shock_index['composite_shock_index']:.2f}")
            if self.shock_index['composite_shock_index'] > 5:
                report_lines.append("Impact level: Severe shock")
            elif self.shock_index['composite_shock_index'] > 3:
                report_lines.append("Impact level: Moderate shock")
            elif self.shock_index['composite_shock_index'] > 1:
                report_lines.append("Impact level: Mild shock")
            else:
                report_lines.append("Impact level: No significant shock")
        
        # 5. Abnormal volatility detection
        report_lines.append("")
        report_lines.append("V. Abnormal Volatility Detection")
        report_lines.append("-" * 40)
        
        if hasattr(self, 'abnormal_detection'):
            report_lines.append(f"Is abnormal: {self.abnormal_detection['is_abnormal']}")
            report_lines.append(f"2σ test: {self.abnormal_detection['2sigma_test']}")
            report_lines.append(f"3x volatility test: {self.abnormal_detection['3x_vol_test']}")
            report_lines.append(f"Percentile rank: {self.abnormal_detection['percentile_rank_pct']:.1f}%")
        
        # 6. Investment recommendations
        report_lines.append("")
        report_lines.append("VI. Investment Recommendations")
        report_lines.append("-" * 40)
        
        if hasattr(self, 'severity_assessment'):
            severity = self.severity_assessment['combined_assessment']
            if "Extreme" in severity or "Major" in severity:
                report_lines.append("Recommendation: Highly cautious, control position size, set strict stop-losses")
                report_lines.append("Rationale: Event impact is major, market volatility elevated")
            elif "Significant" in severity:
                report_lines.append("Recommendation: Participate moderately, diversify risk, monitor subsequent developments")
                report_lines.append("Rationale: Event impact is significant but market reacting orderly")
            else:
                report_lines.append("Recommendation: Trade normally, focus on fundamental changes")
                report_lines.append("Rationale: Event impact is limited, market reaction normal")
        
        report_lines.append("")
        report_lines.append("=" * 80)
        report_lines.append("Analysis Complete")
        report_lines.append("=" * 80)
        
        # Save report
        report_content = "\n".join(report_lines)
        with open('trump_event_analysis_report.txt', 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        # Print report
        print(report_content)
        print("\n✓ Comprehensive analysis report saved to: trump_event_analysis_report.txt")
        
        return report_content
    
    def run_full_analysis(self):
        """Run full analysis pipeline"""
        print("=" * 80)
        print("Trump Shooting Event Full Quantitative Analysis")
        print("=" * 80)
        
        try:
            # 1. Load data
            self.load_and_preprocess_data()
            
            # 2. Calculate technical indicators
            self.calculate_technical_indicators()
            
            # 3. Calculate event day metrics
            self.calculate_event_metrics()
            
            # 4. Calculate shock index
            self.calculate_shock_index()
            
            # 5. Classify event severity (revised)
            self.classify_event_severity()
            
            # 6. Detect abnormal volatility
            self.detect_abnormal_volatility()
            
            # 7. Find extreme volatility dates
            self.find_extreme_volatility_dates()
            
            # 8. Analyze event context
            self.analyze_event_context()
            
            # 9. Generate summary statistics
            self.generate_summary_statistics()
            
            # 10. Save all results
            self.save_results()
            
            print("\n" + "=" * 80)
            print("Analysis pipeline completed!")
            print("=" * 80)
            print("All analysis results saved to the following files:")
            print("1. trump_full_dataset_with_indicators.csv - Full dataset (with technical indicators)")
            print("2. trump_event_metrics.csv - Event day key metrics")
            print("3. trump_shock_index.csv - Event shock index")
            print("4. trump_severity_assessment.csv - Event severity assessment")
            print("5. trump_abnormal_detection.csv - Abnormal volatility detection results")
            print("6. trump_extreme_volatility_dates.csv - Extreme volatility analysis")
            print("7. trump_context_analysis.csv - Event context analysis")
            print("8. trump_summary_statistics.csv - Summary statistics")
            print("9. trump_event_analysis_report.txt - Comprehensive analysis report")
            print("10. top_10_volatile_days.csv - Top 10 volatile days detailed data")
            
            return True
            
        except Exception as e:
            print(f"\nError during analysis: {str(e)}")
            import traceback
            traceback.print_exc()
            return False


# Main program entry
if __name__ == "__main__":
    # Create analyzer instance
    analyzer = TrumpShootingEventAnalyzer()
    
    # Run full analysis
    success = analyzer.run_full_analysis()
    
    if success:
        print("\nAnalysis completed successfully!")
    else:
        print("\nAnalysis failed, please check error messages.")
```